# Cumulative distributions and KS-test

As an additional assessment of the quality of the selected simulations, we compare the cumulative distributions between simulations and observations for $P$, $\dot{P}$ and the fluxes and perform KS-tests. 

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

We perform the following tests:
1. We merge together the samples from the 100 best simulations and consider this as a single sample to compare with observations and compute the KS-test.
2. For each of the 100 best-fitting simulations, we compute the KS statistics between the simulated sample and the observed sample and estimate the fraction of simulations that are compatible with the observed sample (that have p-values larger than 0.01).
3. We compare the distribution of KS statistics obtained from point 2 with the KS statistics computed between random pairs of simulated samples. This procedure allows to check if the discrepancies between the selected simulations and the observations are comparable to the discrepancies expected between different stochastic realizations of the simulator under the best model.

In [ ]:
# import libraries
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import os

import mlpoppyns.simulator.basics.constants as const
from mlpoppyns.simulator.config_simulator import cfg
import utilities.plot_settings

from utilities.load_catalogs import (
    load_atnf_meerkat_catalog,
    load_xray_catalog,
)

from matplotlib.ticker import FuncFormatter

formatter = FuncFormatter(lambda y, _: "{:.16g}".format(y))
from matplotlib.ticker import LogFormatterMathtext

from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde, ks_2samp

In [ ]:
def tage_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar characteristic age from timing properties assuming P >> P0 and constant magnetic field.

    Args:
        P (float): Spin period of a simulated pulsar, measured in [s].
        Pdot (float): Period derivative of a simulated pulsar in [s/s].

    Returns:
        (float): Characteristic age in [yr].
    """

    # Characteristic age definition.
    tage = P / (2 * Pdot) / const.YR_TO_S

    return tage

## Load the observed catalogs

In [ ]:
surveys_atnf, surveys_meerkat = load_atnf_meerkat_catalog(
    "../../data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    "../../data/observations/meerkat_tpa_posselt_2023.csv",
)

In [ ]:
P_pmps_obs = surveys_atnf["PMPS"]["P"]
Pdot_pmps_obs = surveys_atnf["PMPS"]["P_dot"]
S1400_pmps_obs = surveys_meerkat["PMPS"]["S1400"] * 1.e3 # convert from Jy to mJy.

P_smps_obs = surveys_atnf["SMPS"]["P"]
Pdot_smps_obs = surveys_atnf["SMPS"]["P_dot"]
S1400_smps_obs = surveys_meerkat["SMPS"]["S1400"] * 1.e3 # convert from Jy to mJy.

P_htru_obs = surveys_atnf["HTRU_low-mid"]["P"]
Pdot_htru_obs = surveys_atnf["HTRU_low-mid"]["P_dot"]
S1400_htru_obs = surveys_meerkat["HTRU_low-mid"]["S1400"] * 1.e3 # convert from Jy to mJy.

In [ ]:
surveys_xray = load_xray_catalog(
    "../../data/observations/thermal_NS_05-11-2024.csv",
)

In [ ]:
P_x_obs = surveys_xray["P"]
Pdot_x_obs = surveys_xray["P_dot"]
S_x_obs = surveys_xray["S_x_abs"]
age_x_real_obs = surveys_xray["age"]
d_x_real_obs = surveys_xray["dist"]

# Compute the characteristic age in kyr.
age_char_x_obs = tage_from_timing(P_x_obs, Pdot_x_obs) / 1000

# Define the filters for the observed young magnetars and XDINSs.
young_obs_mask = (age_x_real_obs <= 2) | (age_char_x_obs <= 2)
xdins_obs_mask = ((age_x_real_obs >= 5) | (age_char_x_obs >= 5)) & (
    d_x_real_obs <= 0.5
)

In [ ]:
P_x_young_obs = P_x_obs[young_obs_mask]
Pdot_x_young_obs = Pdot_x_obs[young_obs_mask]
S_x_young_obs = S_x_obs[young_obs_mask]

P_xdins_obs = P_x_obs[xdins_obs_mask]
Pdot_xdins_obs = Pdot_x_obs[xdins_obs_mask]
S_xdins_obs = S_x_obs[xdins_obs_mask]

## Load the simulations

We have simulated 100 populations of neutron stars using the best-parameter values sampled from the inferred posterior distribution. 
- When the entire observed X-ray population is considered, we use the trained posterior estimator saved on the PIC at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4`.
- When only the sample of young magnetars and XDINSs is considered for inference, we use the trained posterior estimator saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4`.

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

# By default, we consider the entire X-ray sample to reproduce Figures 3 and 4.
# To consider the inference results using only young magnetars and XDINSs and
# reproduce Figures 7 and 8, we have to set `use_young_xdins_only` to True.
use_young_xdins_only = True

if use_young_xdins_only:
    simulations_path = f"{root_path}/best_simulations_B_double_lognormal_dip-tor_heavy_youngxdins/simulations_best_params/output_simulations"
else:
    simulations_path = (
        f"{root_path}/best_simulations_B_double_lognormal_dip-tor_heavy/simulations_best_params/output_simulations"
    )

# Number of samples in the parsed directory.
n_sim = len(next(os.walk(simulations_path))[1])

print(n_sim)

In [ ]:
P_pmps_sim_list = []
Pdot_pmps_sim_list = []
S1400_pmps_sim_list = []

P_smps_sim_list = []
Pdot_smps_sim_list = []
S1400_smps_sim_list = []

P_htru_sim_list = []
Pdot_htru_sim_list = []
S1400_htru_sim_list = []

P_x_sim_list = []
Pdot_x_sim_list = []
d_x_sim_list = []
S_x_sim_list = []

P_xdins_sim_list = []
Pdot_xdins_sim_list = []
S_xdins_sim_list = []

P_x_young_sim_list = []
Pdot_x_young_sim_list = []
S_x_young_sim_list = []

In [ ]:
# Load simulation data and save the relevant parameters into lists. In particular,
# we need the spin period P, its derivative Pdot, the radio and X-ray fluxes and
# the corresonding logN-logS distributions.

for i in range(n_sim):
    path_to_simulation = f"{simulations_path}/{i:06d}"

    config_json = json.load(
        open(
            pathlib.Path().joinpath(path_to_simulation, "configuration.json"),
        )
    )

    # Skip simulations where the birth rate exceeded the maximum allowed to not bias the birth rate estimate.
    if (
        (config_json["birth_rate_PMPS_at_match"] == 0)
        | (config_json["birth_rate_SMPS_at_match"] == 0)
        | (config_json["birth_rate_HTRU_low_mid_at_match"] == 0)
        | (config_json["birth_rate_xray_realistic_at_match"] == 0)
    ):
        continue

    # Load the `.pkl.gz` files containing the survey results to import.
    df_PMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_PMPS_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_PMPS_sim.head()

    df_SMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_SMPS_results.pkl.gz"
        ),
        compression="gzip",
    )

    df_HTRU_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_x_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_xray_realistic_results.pkl.gz"
        ),
        compression="gzip",
    )

    # Extract relevant quantities.
    P_pmps_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
    Pdot_pmps_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_pmps_sim = df_PMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    P_smps_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
    Pdot_smps_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_smps_sim = df_SMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
    Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_htru_sim = df_HTRU_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    P_x_sim = df_x_sim["P"]["[s]"].to_numpy()
    Pdot_x_sim = df_x_sim["P_dot"]["[s s^-1]"].to_numpy()
    S_x_sim = df_x_sim["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()
    d_x_sim = df_x_sim["dist"]["[kpc]"].to_numpy()
    age_x_sim = df_x_sim["age"]["[yr]"].to_numpy()

    # Filter young magnetars.
    young_mask = age_x_sim <= 2.0e3

    P_x_young_sim = P_x_sim[young_mask]
    Pdot_x_young_sim = Pdot_x_sim[young_mask]
    S_x_young_sim = S_x_sim[young_mask]

    # Filter X-ray emitting NSs with XDINS-like properties.
    xdins_mask = (d_x_sim <= 0.5) & (age_x_sim >= 1.0e5)

    P_xdins_sim = P_x_sim[xdins_mask]
    Pdot_xdins_sim = Pdot_x_sim[xdins_mask]
    S_xdins_sim = S_x_sim[xdins_mask]

    # Append the values to the corresponding lists.
    P_pmps_sim_list.append(P_pmps_sim)
    Pdot_pmps_sim_list.append(Pdot_pmps_sim)
    S1400_pmps_sim_list.append(S1400_pmps_sim)

    P_smps_sim_list.append(P_smps_sim)
    Pdot_smps_sim_list.append(Pdot_smps_sim)
    S1400_smps_sim_list.append(S1400_smps_sim)

    P_htru_sim_list.append(P_htru_sim)
    Pdot_htru_sim_list.append(Pdot_htru_sim)
    S1400_htru_sim_list.append(S1400_htru_sim)

    P_x_sim_list.append(P_x_sim)
    Pdot_x_sim_list.append(Pdot_x_sim)
    S_x_sim_list.append(S_x_sim)

    P_x_young_sim_list.append(P_x_young_sim)
    Pdot_x_young_sim_list.append(Pdot_x_young_sim)
    S_x_young_sim_list.append(S_x_young_sim)

    P_xdins_sim_list.append(P_xdins_sim)
    Pdot_xdins_sim_list.append(Pdot_xdins_sim)
    S_xdins_sim_list.append(S_xdins_sim)

In [ ]:
# Flatten arrays.
P_pmps_sim_all = np.concatenate(P_pmps_sim_list)
Pdot_pmps_sim_all = np.concatenate(Pdot_pmps_sim_list)
S1400_pmps_sim_all = np.concatenate(S1400_pmps_sim_list)

P_smps_sim_all = np.concatenate(P_smps_sim_list)
Pdot_smps_sim_all = np.concatenate(Pdot_smps_sim_list)
S1400_smps_sim_all = np.concatenate(S1400_smps_sim_list)

P_htru_sim_all = np.concatenate(P_htru_sim_list)
Pdot_htru_sim_all = np.concatenate(Pdot_htru_sim_list)
S1400_htru_sim_all = np.concatenate(S1400_htru_sim_list)

P_x_sim_all = np.concatenate(P_x_sim_list)
Pdot_x_sim_all = np.concatenate(Pdot_x_sim_list)
S_x_sim_all = np.concatenate(S_x_sim_list)

P_x_young_sim_all = np.concatenate(P_x_young_sim_list)
Pdot_x_young_sim_all = np.concatenate(Pdot_x_young_sim_list)
S_x_young_sim_all = np.concatenate(S_x_young_sim_list)

P_xdins_sim_all = np.concatenate(P_xdins_sim_list)
Pdot_xdins_sim_all = np.concatenate(Pdot_xdins_sim_list)
S_xdins_sim_all = np.concatenate(S_xdins_sim_list)

## Plot cumulative distributions and perform KS-tests

In [ ]:
def validity(p):
    """
    If the KS-test p-value is larger than 0.01 we consider the null hypothesis that the two sample are drawn from the same distribution valid.
    """
    return "VALID (p > 0.01)" if p > 0.01 else "NOT VALID (p ≤ 0.01)"

In [ ]:
def cumulative_interp_percentiles(x_param_list):
    """
    Calculate percentile envelopes of normalized cumulative curves.

    For each simulation, an interpolated cumulative curve is computed on a common grid. 
    The 2.5th, 50th, and 97.5th percentiles are then calculated across all simulations.

    Args:
        x_list (list of array-like):
            List containing the x values from each simulation.

    Returns:
        tuple of numpy.ndarray:
            A tuple containing:
            - low_cdf_x: 2.5th percentile of the cumulative distributions.
            - median_cdf_x: 50th percentile (median) of the cumulative
              distributions.
            - high_cdf_x: 97.5th percentile of the cumulative
              distributions.
            - x_grid: the grid of x values where the interpolation has been performed.
    """
    # Keep only non-empty simulations for determining the common grid.
    non_empty_x = [x for x in x_param_list if len(x) > 0]
    
    if len(non_empty_x) == 0:
        raise ValueError("All x arrays are empty.")
    
    # Determine the common x range.
    xmin = min(x.min() for x in non_empty_x)
    xmax = max(x.max() for x in non_empty_x)
    
    # Define a common grid.
    x_grid = np.logspace(np.log10(xmin), np.log10(xmax), 1000)
    
    cdf_interp = []
    
    for x in x_param_list:
        if len(x) > 0:
            x_sorted = np.sort(x)
            cdf = np.arange(1, len(x_sorted) + 1) / len(x_sorted)
    
            f = interp1d(
                x_sorted,
                cdf,
                bounds_error=False,
                fill_value=(0.0, np.nan),
            )
    
            cdf_interp.append(f(x_grid))
    
        else:
            # Empty simulation contributes NaN everywhere.
            cdf_interp.append(np.full_like(x_grid, np.nan))
    
    cdf_interp = np.vstack(cdf_interp)
    
    # Compute percentiles while ignoring NaNs.
    low_cdf_x = np.nanpercentile(cdf_interp, 2.5, axis=0)
    median_cdf_x = np.nanpercentile(cdf_interp, 50, axis=0)
    high_cdf_x = np.nanpercentile(cdf_interp, 97.5, axis=0)
    
    return low_cdf_x, median_cdf_x, high_cdf_x, x_grid

In [ ]:
def evaluate_ks_sim_obs(
    P_sim_list,
    P_obs,
    Pdot_sim_list,
    Pdot_obs,
    S_sim_list,
    S_obs,
    p_threshold=0.01,
):
    """Evaluate KS tests between simulated and observed distributions.

    Args:
        P_sim_list (list): List of simulated period distributions.
        P_obs (array-like): Observed period distribution.
        Pdot_sim_list (list): List of simulated period-derivative
        distributions.
        Pdot_obs (array-like): Observed period-derivative distribution.
        S_sim_list (list): List of simulated flux distributions.
        S_obs (array-like): Observed flux distribution.
        p_threshold (float, optional): P-value threshold for determining
        compatibility. Defaults to 0.01.
    
    Returns:
        dict: Dictionary containing KS statistics, p-values, percentile
        summaries, and compatibility fractions.
    """

    # Store KS statistics and p-values
    ks_results = {
        "KS_P": [],
        "p_P": [],
        "KS_Pdot": [],
        "p_Pdot": [],
        "KS_S": [],
        "p_S": [],
    }

    # Period
    for P in P_sim_list:
        if len(P) != 0:
            ksP, pP = ks_2samp(P, P_obs)
            ks_results["KS_P"].append(ksP)
            ks_results["p_P"].append(pP)

    # Period derivative
    for Pdot in Pdot_sim_list:
        if len(Pdot) != 0:
            ksPd, pPd = ks_2samp(Pdot, Pdot_obs)
            ks_results["KS_Pdot"].append(ksPd)
            ks_results["p_Pdot"].append(pPd)

    # Flux
    for S in S_sim_list:
        if len(S) != 0:
            ksS, pS = ks_2samp(S, S_obs)
            ks_results["KS_S"].append(ksS)
            ks_results["p_S"].append(pS)

    # KS statistic percentiles
    ks_summary = {
        "P": {
            "low": np.percentile(ks_results["KS_P"], 2.5),
            "median": np.percentile(ks_results["KS_P"], 50.0),
            "high": np.percentile(ks_results["KS_P"], 97.5),
        },
        "Pdot": {
            "low": np.percentile(ks_results["KS_Pdot"], 2.5),
            "median": np.percentile(ks_results["KS_Pdot"], 50.0),
            "high": np.percentile(ks_results["KS_Pdot"], 97.5),
        },
        "S": {
            "low": np.percentile(ks_results["KS_S"], 2.5),
            "median": np.percentile(ks_results["KS_S"], 50.0),
            "high": np.percentile(ks_results["KS_S"], 97.5),
        },
    }

    # Compatibility fractions
    compatible_P = np.array(ks_results["p_P"]) > p_threshold
    compatible_Pdot = np.array(ks_results["p_Pdot"]) > p_threshold
    compatible_S = np.array(ks_results["p_S"]) > p_threshold

    fraction_compatible = {
        "P": np.mean(compatible_P),
        "Pdot": np.mean(compatible_Pdot),
        "S": np.mean(compatible_S),
        "all": np.mean(
            compatible_P & compatible_Pdot & compatible_S
        ),
    }

    # Print results
    print(
        f"  Period : KS = {ks_summary['P']['median']:.3f} "
        f"+ {ks_summary['P']['high'] - ks_summary['P']['median']:.3f} "
        f"- {ks_summary['P']['median'] - ks_summary['P']['low']:.3f}, "
        f"fraction compatible (p > {p_threshold}) = "
        f"{fraction_compatible['P']:.3f}"
    )

    print(
        f"  Pdot   : KS = {ks_summary['Pdot']['median']:.3f} "
        f"+ {ks_summary['Pdot']['high'] - ks_summary['Pdot']['median']:.3f} "
        f"- {ks_summary['Pdot']['median'] - ks_summary['Pdot']['low']:.3f}, "
        f"fraction compatible (p > {p_threshold}) = "
        f"{fraction_compatible['Pdot']:.3f}"
    )

    print(
        f"  Flux   : KS = {ks_summary['S']['median']:.3f} "
        f"+ {ks_summary['S']['high'] - ks_summary['S']['median']:.3f} "
        f"- {ks_summary['S']['median'] - ks_summary['S']['low']:.3f}, "
        f"fraction compatible (p > {p_threshold}) = "
        f"{fraction_compatible['S']:.3f}\n"
    )

    print(
        f"fraction compatible (all p > {p_threshold}) = "
        f"{fraction_compatible['all']:.3f}\n"
    )

    return {
        "ks_results": ks_results,
        "ks_summary": ks_summary,
        "fraction_compatible": fraction_compatible,
    }

In [ ]:
def evaluate_ks_sim_sim(
    P_sim_list,
    Pdot_sim_list,
    S_sim_list,
):
    """Calculate pairwise KS statistics for simulated pulsar properties.

    For every unique pair of simulations, this function calculates the
    two-sample Kolmogorov-Smirnov (KS) statistic for period, period
    derivative (Pdot), and 1400 MHz flux density. It then summarizes the
    resulting distributions using the 2.5th, 50th, and 97.5th percentiles.

    Args:
        P_sim_list: List of arrays containing simulated spin periods
            for each simulation.
        Pdot_sim_list: List of arrays containing simulated period
            derivatives for each simulation.
        S_sim_list: List of arrays containing simulated 1400 MHz
            flux densities for each simulation.

    Returns:
        dict: Dictionary containing the KS statistic distributions and
        their summary statistics. The dictionary has the following keys:

            - ``KS_P``: Pairwise KS statistics for period.
            - ``KS_Pdot``: Pairwise KS statistics for period derivative.
            - ``KS_S``: Pairwise KS statistics for 1400 MHz flux density.
            - ``KS_P_percentiles``: 2.5th, 50th, and 97.5th percentiles
              for the period KS statistics.
            - ``KS_Pdot_percentiles``: 2.5th, 50th, and 97.5th percentiles
              for the period-derivative KS statistics.
            - ``KS_S_percentiles``: 2.5th, 50th, and 97.5th percentiles
              for the flux-density KS statistics.
    """
    ks_results = {
        "KS_P": [],
        "KS_Pdot": [],
        "KS_S": [],
    }

    for i in range(len(P_sim_list)):
        if len(P_sim_list[i]) > 0:
            for j in range(i + 1, len(P_sim_list)):
                if len(P_sim_list[j]) > 0:
                    ks_results["KS_P"].append(
                        ks_2samp(
                            P_sim_list[i],
                            P_sim_list[j],
                        ).statistic
                    )
        
                    ks_results["KS_Pdot"].append(
                        ks_2samp(
                            Pdot_sim_list[i],
                            Pdot_sim_list[j],
                        ).statistic
                    )
        
                    ks_results["KS_S"].append(
                        ks_2samp(
                            S_sim_list[i],
                            S_sim_list[j],
                        ).statistic
                    )

    # Calculate the 2.5th, 50th, and 97.5th percentiles.
    ks_results["KS_P_percentiles"] = np.percentile(
        ks_results["KS_P"], [2.5, 50.0, 97.5]
    )

    ks_results["KS_Pdot_percentiles"] = np.percentile(
        ks_results["KS_Pdot"], [2.5, 50.0, 97.5]
    )

    ks_results["KS_S_percentiles"] = np.percentile(
        ks_results["KS_S"], [2.5, 50.0, 97.5]
    )

    # Print results.
    ksP_low, ksP_median, ksP_high = ks_results["KS_P_percentiles"]
    ksPd_low, ksPd_median, ksPd_high = ks_results["KS_Pdot_percentiles"]
    ksS_low, ksS_median, ksS_high = ks_results["KS_S_percentiles"]

    print(
        f"  Period      : KS = {ksP_median:.3f} "
        f"+ {ksP_high - ksP_median:.3f} "
        f"- {ksP_median - ksP_low:.3f}"
    )

    print(
        f"  Pdot        : KS = {ksPd_median:.3f} "
        f"+ {ksPd_high - ksPd_median:.3f} "
        f"- {ksPd_median - ksPd_low:.3f}"
    )

    print(
        f"  Flux        : KS = {ksS_median:.3f} "
        f"+ {ksS_high - ksS_median:.3f} "
        f"- {ksS_median - ksS_low:.3f}\n"
    )

    return ks_results

## PMPS

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_pmps_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:blue", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_pmps_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:blue",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S1400_pmps_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:blue", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_pmps_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# Remove observed pulsars without Pdot measurements.
Pdot_pmps_obs = Pdot_pmps_obs[~np.isnan(Pdot_pmps_obs)]

Pdot_sorted = np.sort(Pdot_pmps_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S1400_pmps_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)


# axP.set_xticks([ 1, 10])
# axPdot.set_xticks([1e-11,1e-10,1e-9])
# axS.set_xticks([1e-13,1e-11,1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{1400}$ [mJy]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-2, 1e2)
axPdot.set_xlim(1e-20, 1e-8)
axS.set_xlim(1e-2, 1e4)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

We first merge together all simulations and compute the KS statistics between the merged sample and the observed sample.
Note that in this way, the considered simulated sample is much bigger than the observed one. This makes the empirical simulated CDF extremely precise. Consequently, the KS test can become very sensitive to small differences between the merged simulation distribution and the observed distribution.

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_pmps_sim_all, P_pmps_obs)
ksPd, pPd = ks_2samp(Pdot_pmps_sim_all, Pdot_pmps_obs)
ksS, pS = ks_2samp(S1400_pmps_sim_all, S1400_pmps_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

We now perform the KS-test for each simulation realization against the observed sample and report the median KS-statistics with the 95\% percentile interval.
We also report the fraction of simulation whose p-value is lower than 0.01, i.e., that reject the null hypothesis that the two samples are drawn from the same distribution.

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_pmps_sim_list, P_pmps_obs, Pdot_pmps_sim_list, Pdot_pmps_obs, S1400_pmps_sim_list, S1400_pmps_obs,)

The selected best-model simulations reproduce the observed spin-period and period-derivative distributions with moderate success, with 45\% and 63\% of the realizations, respectively, yielding KS-test p-values above 0.01. The agreement is poorer for the radio-flux distribution, for which only 20\% of the realizations satisfy this criterion. 

We now perform the KS-test between simulated sample pairs to compute the instrinsic variance in our best model, i.e. estimate how different our simulated samples can be between each other.

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_pmps_sim_list, Pdot_pmps_sim_list, S1400_pmps_sim_list)

By comparing these values with the ones computed above we note that for $P$ and $\dot{P}$, the discrepancy between the observations and the simulations is comparable to the intrinsic discrepancy between independent simulation realizations. On the other hand for the radio flux, the observation–simulation discrepancy is approximately 1.8 time larger, indicating that the best model struggles to reproduce its distribution.

## SMPS

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_smps_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:blue", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_smps_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:blue",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S1400_smps_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:blue", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_smps_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# Remove observed pulsars without Pdot measurements.
Pdot_smps_obs = Pdot_smps_obs[~np.isnan(Pdot_smps_obs)]

Pdot_sorted = np.sort(Pdot_smps_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S1400_smps_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)


# axP.set_xticks([ 1, 10])
# axPdot.set_xticks([1e-11,1e-10,1e-9])
# axS.set_xticks([1e-13,1e-11,1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{1400}$ [mJy]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-2, 1e2)
axPdot.set_xlim(1e-20, 1e-8)
axS.set_xlim(1e-2, 1e4)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_smps_sim_all, P_smps_obs)
ksPd, pPd = ks_2samp(Pdot_smps_sim_all, Pdot_smps_obs)
ksS, pS = ks_2samp(S1400_smps_sim_all, S1400_smps_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_smps_sim_list, P_smps_obs, Pdot_smps_sim_list, Pdot_smps_obs, S1400_smps_sim_list, S1400_smps_obs,)

The selected best-model simulations reproduce the observed spin-period and period-derivative distributions with moderate success, with 75\% and 58\% of the realizations, respectively, yielding KS-test p-values above 0.01. The agreement is a bit poorer for the radio-flux distribution, for which, 32\% of the realizations satisfy this criterion. 

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_smps_sim_list, Pdot_smps_sim_list, S1400_smps_sim_list)

By comparing these values with the ones computed above we note that for $P$ and $\dot{P}$, the discrepancy between the observations and the simulations is comparable to the intrinsic discrepancy between independent simulation realizations. On the other hand for the radio flux, the observation–simulation discrepancy is approximately 2.2 time larger, indicating that the best model struggles to reproduce its distribution.

## HTRU

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_htru_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:blue", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_htru_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:blue",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S1400_htru_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:blue", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:blue",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_htru_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# Remove observed pulsars without Pdot measurements.
Pdot_htru_obs = Pdot_htru_obs[~np.isnan(Pdot_htru_obs)]

Pdot_sorted = np.sort(Pdot_htru_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S1400_htru_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)


# axP.set_xticks([ 1, 10])
# axPdot.set_xticks([1e-11,1e-10,1e-9])
# axS.set_xticks([1e-13,1e-11,1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{1400}$ [mJy]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-2, 1e2)
axPdot.set_xlim(1e-20, 1e-8)
axS.set_xlim(1e-2, 1e4)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_htru_sim_all, P_htru_obs)
ksPd, pPd = ks_2samp(Pdot_htru_sim_all, Pdot_htru_obs)
ksS, pS = ks_2samp(S1400_htru_sim_all, S1400_htru_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_htru_sim_list, P_htru_obs, Pdot_htru_sim_list, Pdot_htru_obs, S1400_htru_sim_list, S1400_htru_obs,)

The selected best-model simulations reproduce the observed $P$, $\dot{P}$ and flux distributions with moderate success, with 45\% and 60\%  and 86\% of the realizations, respectively, yielding KS-test p-values above 0.01.

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_htru_sim_list, Pdot_htru_sim_list, S1400_htru_sim_list)

By comparing these values with the ones computed above we note that for all quantities, the discrepancy between the observations and the simulations is comparable to the intrinsic discrepancy between independent simulation realizations.

## Young magnetars

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_x_young_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:pink", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:pink",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_x_young_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:pink",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:pink",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S_x_young_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:pink", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:pink",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_x_young_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

Pdot_sorted = np.sort(Pdot_x_young_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S_x_young_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)

axPdot.set_xticks([1e-11, 1e-10, 1e-9])
axP.set_xticks([1, 10])
axS.set_xticks([1e-13, 1e-11, 1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-1, 1e2)
axPdot.set_xlim(1e-12, 1e-8)
axS.set_xlim(1e-15, 1e-9)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_x_young_sim_all, P_x_young_obs)
ksPd, pPd = ks_2samp(Pdot_x_young_sim_all, Pdot_x_young_obs)
ksS, pS = ks_2samp(S_x_young_sim_all, S_x_young_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_x_young_sim_list, P_x_young_obs, Pdot_x_young_sim_list, Pdot_x_young_obs, S_x_young_sim_list, S_x_young_obs,)

The selected best-model simulations reproduce the observed $P$, $\dot{P}$ and flux distributions with success, with 98\% and 88\% and 100\% of the realizations, respectively, yielding KS-test p-values above 0.01.

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_x_young_sim_list, Pdot_x_young_sim_list, S_x_young_sim_list)

By comparing these values with the ones computed above we note that for all quantities, the discrepancy between the observations and the simulations is comparable to the intrinsic discrepancy between independent simulation realizations.

## XDINS

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_xdins_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:orange", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:orange",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_xdins_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:orange",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:orange",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S_xdins_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:orange", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:orange",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_xdins_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

Pdot_sorted = np.sort(Pdot_xdins_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S_xdins_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)

axPdot.set_xticks([1e-15, 1e-12, 1e-9])
axP.set_xticks([1, 10, 1e2])
axS.set_xticks([1e-13, 1e-11, 1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-1, 1e2)
axPdot.set_xlim(1e-16, 1e-8)
axS.set_xlim(1e-15, 1e-9)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_xdins_sim_all, P_xdins_obs)
ksPd, pPd = ks_2samp(Pdot_xdins_sim_all, Pdot_xdins_obs)
ksS, pS = ks_2samp(S_xdins_sim_all, S_xdins_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_xdins_sim_list, P_xdins_obs, Pdot_xdins_sim_list, Pdot_xdins_obs, S_xdins_sim_list, S_xdins_obs,)

The selected best-model simulations reproduce the observed $P$, $\dot{P}$ and flux distributions with success, with 97\% and 95\% and 100\% of the realizations, respectively, yielding KS-test p-values above 0.01.

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_xdins_sim_list, Pdot_xdins_sim_list, S_xdins_sim_list)

By comparing these values with the ones computed above we note that for all quantities, the discrepancy between the observations and the simulations is comparable to the intrinsic discrepancy between independent simulation realizations.

## Full X-ray sample

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)  # smaller = less horizontal space
axP, axPdot, axS = axes

# ===================================
#     SIMULATED CUMULATIVE CURVES
# ===================================

# Period.
low_cdf_P, median_cdf_P, high_cdf_P, P_grid = cumulative_interp_percentiles(P_x_sim_list)

axP.plot(
    P_grid, median_cdf_P, color="tab:purple", lw=4, label="Median"
)
axP.fill_between(
    P_grid,
    low_cdf_P,
    high_cdf_P,
    color="tab:purple",
    alpha=0.3,
    label="95 % C.I.",
)

# Period derivative.
low_cdf_Pdot, median_cdf_Pdot, high_cdf_Pdot, Pdot_grid = cumulative_interp_percentiles(Pdot_x_sim_list)

axPdot.plot(
    Pdot_grid,
    median_cdf_Pdot,
    color="tab:purple",
    lw=4,
    label="Median",
)
axPdot.fill_between(
    Pdot_grid,
    low_cdf_Pdot,
    high_cdf_Pdot,
    color="tab:purple",
    alpha=0.3,
    label="95 % C.I,",
)

# Flux.
low_cdf_S, median_cdf_S, high_cdf_S, S_grid = cumulative_interp_percentiles(S_x_sim_list)

axS.plot(
    S_grid, median_cdf_S, color="tab:purple", lw=4, label="Median"
)
axS.fill_between(
    S_grid,
    low_cdf_S,
    high_cdf_S,
    color="tab:purple",
    alpha=0.3,
    label="95 % C.I,",
)
# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

P_sorted = np.sort(P_x_obs)
cdf = np.arange(1, len(P_sorted) + 1) / len(P_sorted)

axP.plot(
    P_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

Pdot_sorted = np.sort(Pdot_x_obs)
cdf = np.arange(1, len(Pdot_sorted) + 1) / len(Pdot_sorted)

axPdot.plot(
    Pdot_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

S_sorted = np.sort(S_x_obs)
cdf = np.arange(1, len(S_sorted) + 1) / len(S_sorted)

axS.plot(
    S_sorted,
    cdf,
    color="dimgrey",
    lw=4,
    label="Observed",
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both", labelsize=35)
    ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)

axPdot.set_xticks([1e-15, 1e-13, 1e-11, 1e-9])
axP.set_xticks([1, 10])
axS.set_xticks([1e-13, 1e-11, 1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22,
)
axP.set_xlim(1e-1, 1e2)
axPdot.set_xlim(1e-16, 1e-9)
axS.set_xlim(1e-15, 1e-9)


axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
print("\n=======================================================")
print(" K–S TEST RESULTS (Merged simulations vs observations)")
print("=======================================================\n")

# Evaluate KS tests.
ksP, pP = ks_2samp(P_x_sim_all, P_x_obs)
ksPd, pPd = ks_2samp(Pdot_x_sim_all, Pdot_x_obs)
ksS, pS = ks_2samp(S_x_sim_all, S_x_obs)

# Print results.
print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs observations)")
print("===========================================================\n")

_ = evaluate_ks_sim_obs(P_x_sim_list, P_x_obs, Pdot_x_sim_list, Pdot_x_obs, S_x_sim_list, S_x_obs,)

The selected best-model simulations reproduce the observed $\dot{P}$ and flux distributions with reasonable success, with 69\% and 97\% of the realizations, respectively, yielding KS-test p-values above 0.01. The agreement is poorer for the spin period distribution, for which, 18\% of the realizations satisfy this criterion. 

In [ ]:
print("\n===========================================================")
print(" K–S TEST RESULTS (Individual simulations vs simulations)")
print("===========================================================\n")

_ = evaluate_ks_sim_sim(P_x_sim_list, Pdot_x_sim_list, S_x_sim_list)

By comparing these values with the ones computed above we note that for all quantities, the discrepancy between the observations and the simulations is larger than the intrinsic discrepancy between independent simulation realizations. This is an indication that the best model struggle to reproduce the full X-ray population.